# Taxonomia de erros em questões

Notebook organizado para Colab com dois fluxos:

1. **Avaliação e reformulação de questões existentes** a partir de `rejected_questions.csv`.
2. **Geração de novas questões** com revisão dupla, validações estruturais, checagem simbólica quando aplicável, resolvedores cegos, rubrica e iteração.

Regra editorial fixa: uma questão pode usar afirmações ou asserções quando o formato pedir, mas nunca os dois formatos ao mesmo tempo.

As chaves de API foram mantidas no notebook, conforme solicitado. Depois do teste, revogue as chaves usadas.


## Como rodar

1. Suba este notebook no Google Colab.
2. Rode as células em ordem.
3. Quando solicitado, envie o arquivo `rejected_questions.csv`.
4. Para teste rápido, a geração de novas questões está limitada a `N_QUESTOES = 2`.
5. Para rodar a geração completa, altere `N_QUESTOES` para `60`.


## 1. Instalação e configuração

Esta célula instala as bibliotecas usadas no notebook: OpenAI, pandas, Pydantic, SymPy e ReportLab.


In [ ]:
!pip -q install openai pandas pydantic tqdm tenacity sympy reportlab


## 2. Imports, clientes e parâmetros gerais

Aqui ficam os imports, os clientes de API e os parâmetros que controlam custo e tamanho do teste.


In [ ]:
import os
import re
import json
import time
import uuid
import math
import html
import traceback
import unicodedata
from pathlib import Path
from typing import Any, Dict, List, Literal, Optional

import pandas as pd
from tqdm.auto import tqdm
from pydantic import BaseModel, Field, ConfigDict, field_validator
from openai import OpenAI
import openai

import sympy as sp
from sympy.parsing.sympy_parser import (
    parse_expr,
    standard_transformations,
    implicit_multiplication_application,
    convert_xor,
)

try:
    from google.colab import files
except Exception:
    files = None


pd.set_option("display.max_colwidth", 180)


# =========================
# 1. Clientes
# =========================

client_gpt = OpenAI(
    api_key= "")

client_sabia = openai.OpenAI(
    api_key='',
    base_url="https://chat.maritaca.ai/api",
)


# Alias usado pelo pipeline robusto de geração.
client_openai = client_gpt


# Parametros da parte 1: avaliacao/reformulacao de questoes existentes.
N_AVALIAR_EXISTENTES = None  # None = todas as questoes filtradas; use um inteiro para teste rapido.
SLEEP_ENTRE_CHAMADAS = 1.0


# Parametros da parte 2: geracao de novas questoes.
N_QUESTOES = 2  # teste rapido. Troque para 60 quando quiser rodar o plano completo.
LIMIAR_APROVACAO = 95
MAX_REVISOES = 5


BASE_DIR = Path("/content") if Path("/content").exists() else Path("")

ARQUIVO_AVALIACOES = BASE_DIR / "problemas_matematica_gpt52_sabia4.csv"
ARQUIVO_REVISAO_JSON = BASE_DIR / "revisao_questoes.json"
ARQUIVO_REVISAO_CSV = BASE_DIR / "revisao_questoes.csv"
ARQUIVO_REVISADAS = BASE_DIR / "questoes_revisadas.csv"
ARQUIVO_COMPARACAO_PDF = BASE_DIR / "comparacao_questoes.pdf"

ARQ_JSON = BASE_DIR / "questoes_geradas_pipeline_robusto.json"
ARQ_CSV = BASE_DIR / "questoes_geradas_pipeline_robusto.csv"


MODEL_AVALIADOR_GPT = "o3-mini"
MODEL_COMITE_EXISTENTES = "sabia-4"

MODEL_AUTOR = "sabia-4"
MODEL_REVISOR_OPENAI = "o3-mini"
MODEL_COMITE = "sabia-4"
MODEL_AUDITOR = "sabia-4"
MODEL_RESOLVEDOR = "o3-mini"
MODEL_SABIA = "sabia-4"

PROVEDOR_AUTOR = "sabia"
PROVEDOR_REVISOR_OPENAI = "openai"
PROVEDOR_COMITE = "sabia"
PROVEDOR_AUDITOR = "sabia"
PROVEDOR_RESOLVEDOR = "openai"

REINICIAR_GERACAO = True

CAMPOS_AFIRMACOES = [
    "statement_i",
    "statement_ii",
    "statement_iii",
    "statement_iv",
]

CAMPOS_ASSERCOES = [
    "assertion_i",
    "assertion_ii",
]


def _tem_conteudo(valor):
    return str(valor or "").strip() != ""


def normalizar_afirmacoes_assercoes_dict(q):
    q = dict(q or {})
    tem_afirmacoes = any(_tem_conteudo(q.get(campo, "")) for campo in CAMPOS_AFIRMACOES)
    tem_assercoes = any(_tem_conteudo(q.get(campo, "")) for campo in CAMPOS_ASSERCOES)

    if tem_afirmacoes and tem_assercoes:
        tipo = remover_acentos(q.get("question_type", "")).lower()
        peso_afirmacoes = sum(len(str(q.get(campo, "") or "").strip()) for campo in CAMPOS_AFIRMACOES)
        peso_assercoes = sum(len(str(q.get(campo, "") or "").strip()) for campo in CAMPOS_ASSERCOES)
        manter_assercoes = "asserc" in tipo or peso_assercoes > peso_afirmacoes
        limpar = CAMPOS_AFIRMACOES if manter_assercoes else CAMPOS_ASSERCOES
        for campo in limpar:
            q[campo] = ""

    return q


## 3. Entrada de dados

O fluxo abaixo espera o arquivo `rejected_questions.csv`. Se ele ainda não estiver no ambiente do Colab, a célula abre o seletor de upload.


In [ ]:
if not Path("rejected_questions.csv").exists():
    if files is None:
        raise FileNotFoundError("Coloque rejected_questions.csv no diretorio de execucao.")

    print("Envie o arquivo rejected_questions.csv")
    files.upload()

df = pd.read_csv("rejected_questions.csv")
print("Linhas carregadas:", len(df))
df.head()


## 4. Filtro das disciplinas de Matemática

Esta etapa corrige diferenças simples de acentuação/encoding no nome das disciplinas e mantém apenas as áreas usadas no estudo.


In [ ]:
def corrigir_mojibake(texto):
    if pd.isna(texto):
        return ""

    texto = str(texto).strip()
    try:
        corrigido = texto.encode("latin1").decode("utf-8")
        if "Ã" in texto or "�" in texto:
            texto = corrigido
    except Exception:
        pass

    return texto.strip()


def remover_acentos(texto):
    return "".join(
        ch
        for ch in unicodedata.normalize("NFD", str(texto or ""))
        if unicodedata.category(ch) != "Mn"
    )


def chave_disciplina(texto):
    return re.sub(r"\s+", " ", remover_acentos(corrigir_mojibake(texto)).upper()).strip()


DISCIPLINAS_ALVO = {
    "CALCULO DIFERENCIAL E INTEGRAL II": "Cálculo Diferencial e Integral II",
    "CALCULO DIFERENCIAL E INTEGRAL III": "Cálculo Diferencial e Integral III",
    "ESTRUTURAS ALGEBRICAS": "Estruturas Algébricas",
    "GEOMETRIA ESPACIAL": "Geometria Espacial",
}


df = df.copy()
df["discipline_key"] = df["discipline"].apply(chave_disciplina)

df_math = df[df["discipline_key"].isin(DISCIPLINAS_ALVO)].copy()
df_math["discipline"] = df_math["discipline_key"].map(DISCIPLINAS_ALVO)
df_math = df_math.drop(columns=["discipline_key"])

print("Questoes de matematica:", df_math.shape)
df_math["discipline"].value_counts()


---
# Parte A: avaliação e reformulação de questões existentes


## 5. Avaliador cego

Cada questão filtrada é enviada para dois avaliadores independentes:

- GPT-5.2;
- Sabiá-4.

Cada avaliador devolve apenas uma lista de problemas em JSON.


In [ ]:
# =========================
# 2. Montar questão
# =========================

def texto_valido(x):
    return pd.notna(x) and str(x).strip() != ""


def montar_questao(row):
    partes = []

    partes.append(f"ID da questão: {row.get('question_id', '')}")
    partes.append(f"Disciplina: {row.get('discipline', '')}")
    partes.append(f"Tipo de questão: {row.get('question_type', '')}")
    partes.append(f"Nível taxonômico declarado: {row.get('taxonomy_level', '')}")

    if texto_valido(row.get("base_text")):
        partes.append(f"\nTexto-base:\n{row.get('base_text')}")

    if texto_valido(row.get("stem")):
        partes.append(f"\nEnunciado:\n{row.get('stem')}")

    campos = [
        ("statement_i", "I"),
        ("statement_ii", "II"),
        ("statement_iii", "III"),
        ("statement_iv", "IV"),
        ("assertion_i", "Asserção I"),
        ("assertion_ii", "Asserção II"),
    ]

    for col, label in campos:
        if texto_valido(row.get(col)):
            partes.append(f"\n{label}:\n{row.get(col)}")

    partes.append("\nAlternativas:")

    alternativas = [
        ("option_a", "A"),
        ("option_b", "B"),
        ("option_c", "C"),
        ("option_d", "D"),
        ("option_e", "E"),
    ]

    for col, label in alternativas:
        if texto_valido(row.get(col)):
            partes.append(f"{label}) {row.get(col)}")

    if texto_valido(row.get("correct_option")):
        partes.append(f"\nGabarito informado: {row.get('correct_option')}")

    return "\n".join(partes)


# =========================
# 3. Limpeza de JSON
# =========================

def extrair_json(texto):
    """
    Tenta extrair JSON mesmo quando o modelo devolve ```json ... ```.
    """
    if texto is None:
        return None

    texto = texto.strip()

    texto = re.sub(r"^```json", "", texto, flags=re.IGNORECASE).strip()
    texto = re.sub(r"^```", "", texto).strip()
    texto = re.sub(r"```$", "", texto).strip()

    try:
        return json.loads(texto)
    except Exception:
        pass

    match = re.search(r"\{.*\}", texto, flags=re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            return None

    return None


def normalizar_objeto(obj):
    """
    Garante que a saída final tenha sempre o formato:
    {"problemas": [...]}
    """
    if obj is None:
        return {"problemas": []}

    if isinstance(obj, dict) and "problemas" in obj:
        problemas = obj["problemas"]
        if isinstance(problemas, list):
            return {"problemas": [str(p) for p in problemas]}
        else:
            return {"problemas": [str(problemas)]}

    return {"problemas": [json.dumps(obj, ensure_ascii=False)]}


# =========================
# 4. Prompt
# =========================

PROMPT_SISTEMA = """
Você é um especialista em avaliação educacional e elaboração de questões de Matemática.

Sua tarefa é identificar todos os problemas pedagógicos, conceituais, formais e matemáticos de uma questão gerada por IA.

Considere:
- erro conceitual;
- erro de cálculo;
- gabarito incorreto;
- mais de uma alternativa correta;
- ausência de alternativa correta;
- enunciado ambíguo;
- texto-base irrelevante ou insuficiente;
- distratores fracos;
- alternativa com pista formal;
- incompatibilidade com o nível taxonômico declarado;
- notação matemática inadequada;
- questão superficial.

Retorne SOMENTE um JSON válido, exatamente neste formato:

{
  "problemas": [
    "...",
    "...",
    "..."
  ]
}

Não use markdown.
Não use ```json.
Não inclua explicações fora do JSON.
"""


# =========================
# 5. Funções GPT e Sabiá
# =========================

def avaliar_problemas_gpt(row):
    texto_questao = montar_questao(row)

    resposta = client_gpt.chat.completions.create(
        model=MODEL_AVALIADOR_GPT,
        messages=[
            {"role": "system", "content": PROMPT_SISTEMA},
            {"role": "user", "content": texto_questao},
        ],
        response_format={"type": "json_object"},
    )

    bruto = resposta.choices[0].message.content
    obj = normalizar_objeto(extrair_json(bruto))

    return {
        "bruto": bruto,
        "json": obj,
        "problemas": obj["problemas"]
    }


def avaliar_problemas_sabia(row):
    texto_questao = montar_questao(row)

    resposta = client_sabia.chat.completions.create(
        model="sabia-4",
        messages=[
            {"role": "system", "content": PROMPT_SISTEMA},
            {"role": "user", "content": texto_questao},
        ],
        max_tokens=1500,
    )

    bruto = resposta.choices[0].message.content
    obj = normalizar_objeto(extrair_json(bruto))

    return {
        "bruto": bruto,
        "json": obj,
        "problemas": obj["problemas"]
    }


# =========================
# 6. Rodar com prints
# =========================

def rodar_avaliacoes(df_math, n=None, sleep=1):
    df_result = df_math.copy()

    resultados_gpt = []
    resultados_sabia = []

    total = len(df_result) if n is None else min(n, len(df_result))

    for i, (idx, row) in enumerate(df_result.head(total).iterrows(), start=1):
        qid = row.get("question_id", idx)

        print("=" * 100)
        print(f"QUESTÃO {i}/{total} | question_id = {qid}")
        print("=" * 100)

        texto_questao = montar_questao(row)
        print("\nQUESTÃO ENVIADA AO MODELO:\n")
        print(texto_questao[:2500])

        print("\n" + "-" * 100)
        print("RESPOSTA o3-mini")
        print("-" * 100)

        try:
            saida_gpt = avaliar_problemas_gpt(row)
            print(saida_gpt["bruto"])
            print("\nProblemas extraídos GPT:")
            for p in saida_gpt["problemas"]:
                print("-", p)
        except Exception as e:
            saida_gpt = {
                "bruto": None,
                "json": {"problemas": []},
                "problemas": [],
                "erro": str(e)
            }
            print("ERRO GPT:", e)

        print("\n" + "-" * 100)
        print("RESPOSTA SABIÁ-4")
        print("-" * 100)

        try:
            saida_sabia = avaliar_problemas_sabia(row)
            print(saida_sabia["bruto"])
            print("\nProblemas extraídos Sabiá:")
            for p in saida_sabia["problemas"]:
                print("-", p)
        except Exception as e:
            saida_sabia = {
                "bruto": None,
                "json": {"problemas": []},
                "problemas": [],
                "erro": str(e)
            }
            print("ERRO SABIÁ:", e)

        resultados_gpt.append(saida_gpt)
        resultados_sabia.append(saida_sabia)

        time.sleep(sleep)

    df_result = df_result.head(total).copy()
    df_result["problemas_gpt_52"] = resultados_gpt
    df_result["problemas_sabia_4"] = resultados_sabia

    return df_result


## 6. Rodar avaliação das questões existentes

O resultado é salvo em `problemas_matematica_gpt52_sabia4.csv`.


In [ ]:
df_math_avaliado = rodar_avaliacoes(
    df_math,
    n=N_AVALIAR_EXISTENTES,
    sleep=SLEEP_ENTRE_CHAMADAS,
)

df_math_avaliado.to_csv(ARQUIVO_AVALIACOES, index=False)
print("Arquivo salvo:", ARQUIVO_AVALIACOES)
df_math_avaliado.head()


## 7. Comitê de revisão

O comitê recebe a questão original e os problemas apontados pelos dois avaliadores. Ele separa concordâncias, divergências, problemas corrigidos e devolve uma nova versão da questão.


In [ ]:
import json
import re
import time


def extrair_json(texto):
    """
    Extrai um JSON mesmo quando o modelo devolve markdown.
    """
    if texto is None:
        return None

    texto = texto.strip()

    texto = re.sub(r"^```json", "", texto, flags=re.IGNORECASE).strip()
    texto = re.sub(r"^```", "", texto).strip()
    texto = re.sub(r"```$", "", texto).strip()

    try:
        return json.loads(texto)
    except Exception:
        pass

    match = re.search(r"\{.*\}", texto, flags=re.DOTALL)

    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            return None

    return None


def revisar_questao_gpt(row, max_retries=3):

    questao = montar_questao(row)

    problemas_gpt = "\n".join(
        row["problemas_gpt_52"]["problemas"]
    )

    problemas_sabia = "\n".join(
        row["problemas_sabia_4"]["problemas"]
    )

    prompt = f"""
Você é um especialista em avaliação educacional e elaboração de questões de Matemática para o Ensino Superior.

Você recebeu uma questão originalmente gerada por IA.

Ela já foi analisada independentemente por dois modelos.

Sua função é atuar como um COMITÊ DE REVISÃO.

=========================
QUESTÃO ORIGINAL
=========================

{questao}

=========================
PROBLEMAS IDENTIFICADOS PELO o3-mini
=========================

{problemas_gpt}

=========================
PROBLEMAS IDENTIFICADOS PELO SABIÁ-4
=========================

{problemas_sabia}

=========================

Realize as seguintes etapas:

1. Identifique os problemas em que ambos os modelos concordam.

2. Identifique os problemas exclusivos do GPT.

3. Identifique os problemas exclusivos do Sabiá.

4. Decida quais problemas realmente precisam ser corrigidos.

5. Reescreva completamente a questão corrigindo esses problemas.

IMPORTANTE

- Preserve o conteúdo matemático.

- Preserve o objetivo pedagógico.

- Preserve a disciplina.

- Preserve o tipo da questão.

- Preserve o nível taxonômico sempre que possível.

- Caso o nível taxonômico esteja inadequado, adapte a questão para realmente atingir esse nível.

- Mantenha apenas UMA alternativa correta.

- Melhore os distratores.

- Corrija linguagem, notação matemática e ambiguidades.

- Pode usar afirmações OU asserções quando o formato da questão exigir, mas nunca os dois formatos na mesma questão.

- Se usar afirmações, deixe assertion_i e assertion_ii vazios.

- Se usar asserções, deixe statement_i, statement_ii, statement_iii e statement_iv vazios.

- Para Múltipla Escolha Simples comum, prefira enunciado direto sem afirmações nem asserções.

Retorne SOMENTE um JSON válido.

Formato obrigatório:

{{
"concordancias":[
"..."
],

"problemas_exclusivos_gpt":[
"..."
],

"problemas_exclusivos_sabia":[
"..."
],

"problemas_corrigidos":[
"..."
],

"justificativa":"",

"questao_revisada":{{

"base_text":"",

"stem":"",

"statement_i":"",

"statement_ii":"",

"statement_iii":"",

"statement_iv":"",

"assertion_i":"",

"assertion_ii":"",

"option_a":"",

"option_b":"",

"option_c":"",

"option_d":"",

"option_e":"",

"correct_option":""

}}

}}

NÃO escreva comentários.

NÃO utilize markdown.

NÃO utilize ```json.

A resposta deve conter APENAS o JSON.
"""

    for tentativa in range(max_retries):

        try:

            print(f"   o3-mini | tentativa {tentativa+1}")

            resposta = client_sabia.chat.completions.create(

                model=MODEL_COMITE_EXISTENTES,

                messages=[
                    {
                        "role":"system",
                        "content":"Você é especialista em avaliação educacional."
                    },
                    {
                        "role":"user",
                        "content":prompt
                    }
                ],

            )

            bruto = resposta.choices[0].message.content

            obj = extrair_json(bruto)

            if obj is not None:

                print("   ✓ JSON válido")

                return obj

            print("   JSON inválido... tentando novamente")

        except Exception as e:

            print(e)

        time.sleep(2)

    print("="*80)
    print("FALHA NA QUESTÃO")
    print("="*80)

    print(bruto if "bruto" in locals() else "")

    return {
        "erro": True,
        "question_id": row["question_id"],
        "concordancias": [],
        "problemas_exclusivos_gpt": [],
        "problemas_exclusivos_sabia": [],
        "problemas_corrigidos": [],
        "justificativa": "",
        "questao_revisada": {}
    }


## 8. Rodar reformulação das questões avaliadas

Esta etapa salva três arquivos:

- `revisao_questoes.json`: registro completo da revisão;
- `revisao_questoes.csv`: resumo das decisões do comitê;
- `questoes_revisadas.csv`: questões reformuladas em formato tabular.


In [ ]:
resultados_revisao = []

for i, (idx, row) in enumerate(df_math_avaliado.iterrows(), start=1):
    print("=" * 100)
    print(f"REVISANDO QUESTAO {i}/{len(df_math_avaliado)} | question_id = {row['question_id']}")
    print("=" * 100)

    revisao = revisar_questao_gpt(row)

    if isinstance(revisao, dict):
        revisao["questao_revisada"] = normalizar_afirmacoes_assercoes_dict(
            revisao.get("questao_revisada", {})
        )

    resultados_revisao.append({
        "question_id": row["question_id"],
        "discipline": row["discipline"],
        "question_type": row["question_type"],
        "taxonomy_level": row["taxonomy_level"],
        "revisao": revisao,
    })

    with open(ARQUIVO_REVISAO_JSON, "w", encoding="utf-8") as f:
        json.dump(resultados_revisao, f, ensure_ascii=False, indent=2)

    linhas_resumo = []
    linhas_questoes = []

    for r in resultados_revisao:
        rev = r["revisao"]
        linhas_resumo.append({
            "question_id": r["question_id"],
            "discipline": r["discipline"],
            "question_type": r["question_type"],
            "taxonomy_level": r["taxonomy_level"],
            "concordancias": " | ".join(rev.get("concordancias", [])),
            "problemas_exclusivos_gpt": " | ".join(rev.get("problemas_exclusivos_gpt", [])),
            "problemas_exclusivos_sabia": " | ".join(rev.get("problemas_exclusivos_sabia", [])),
            "problemas_corrigidos": " | ".join(rev.get("problemas_corrigidos", [])),
            "justificativa": rev.get("justificativa", ""),
        })

        q = normalizar_afirmacoes_assercoes_dict(rev.get("questao_revisada", {}))
        linhas_questoes.append({
            "question_id": r["question_id"],
            "discipline": r["discipline"],
            "question_type": r["question_type"],
            "taxonomy_level": r["taxonomy_level"],
            "base_text": q.get("base_text", ""),
            "stem": q.get("stem", ""),
            "statement_i": q.get("statement_i", ""),
            "statement_ii": q.get("statement_ii", ""),
            "statement_iii": q.get("statement_iii", ""),
            "statement_iv": q.get("statement_iv", ""),
            "assertion_i": q.get("assertion_i", ""),
            "assertion_ii": q.get("assertion_ii", ""),
            "option_a": q.get("option_a", ""),
            "option_b": q.get("option_b", ""),
            "option_c": q.get("option_c", ""),
            "option_d": q.get("option_d", ""),
            "option_e": q.get("option_e", ""),
            "correct_option": q.get("correct_option", ""),
        })

    pd.DataFrame(linhas_resumo).to_csv(ARQUIVO_REVISAO_CSV, index=False)
    pd.DataFrame(linhas_questoes).to_csv(ARQUIVO_REVISADAS, index=False)

    print("Salvo:", ARQUIVO_REVISAO_JSON, ARQUIVO_REVISAO_CSV, ARQUIVO_REVISADAS)
    time.sleep(SLEEP_ENTRE_CHAMADAS)

print("FINALIZADO.")


## 9. Comparação entre original e revisada

Esta etapa junta as versões por `question_id` para inspeção lado a lado.


In [ ]:
df_revisadas = pd.read_csv(ARQUIVO_REVISADAS)

comparacao = df_math.merge(
    df_revisadas,
    on="question_id",
    suffixes=("_original", "_revisada"),
)

colunas_comparacao = [
    "question_id",
    "stem_original",
    "stem_revisada",
    "option_a_original",
    "option_a_revisada",
    "option_b_original",
    "option_b_revisada",
    "option_c_original",
    "option_c_revisada",
    "option_d_original",
    "option_d_revisada",
    "option_e_original",
    "option_e_revisada",
    "correct_option_original",
    "correct_option_revisada",
]

comparacao[colunas_comparacao].head()


## 10. Visualizar uma questão original e sua versão revisada

Por padrão, a célula mostra a primeira questão revisada. Altere `qid` se quiser ver outra.


In [ ]:
def imprimir_questao(row, titulo):
    print("=" * 80)
    print(titulo)
    print("=" * 80)

    campos = [
        ("discipline", "Disciplina"),
        ("question_type", "Tipo"),
        ("taxonomy_level", "Nivel taxonomico"),
        ("base_text", "Texto-base"),
        ("stem", "Enunciado"),
    ]

    for col, nome in campos:
        if col in row.index and pd.notna(row[col]) and str(row[col]).strip() != "":
            print(f"\n{nome}:")
            print(row[col])

    print("\nAlternativas")
    for col, letra in [
        ("option_a", "A"),
        ("option_b", "B"),
        ("option_c", "C"),
        ("option_d", "D"),
        ("option_e", "E"),
    ]:
        if col in row.index and pd.notna(row[col]) and str(row[col]).strip() != "":
            print(f"{letra}) {row[col]}")

    print("\nGabarito:", row["correct_option"])


qid = df_revisadas["question_id"].iloc[0]

linha_original = df_math[df_math["question_id"] == qid].iloc[0]
linha_revisada = df_revisadas[df_revisadas["question_id"] == qid].iloc[0]

imprimir_questao(linha_original, "QUESTAO ORIGINAL")
print("\n\n")
imprimir_questao(linha_revisada, "QUESTAO REVISADA")


## 11. PDF de comparação

Gera um PDF com a versão original e a revisada lado a lado, quando couber na página.


In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, PageBreak, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import cm
import pandas as pd
import html

def clean(x):
    if pd.isna(x):
        return ""
    x = str(x)
    replaces = {
        "\\le": "≤", "\\ge": "≥", "\\neq": "≠",
        "\\in": "∈", "\\alpha": "α", "\\beta": "β",
        "\\gamma": "γ", "\\theta": "θ", "\\pi": "π",
        "\\parallel": "∥"
    }
    for a, b in replaces.items():
        x = x.replace(a, b)
    return html.escape(x)

def questao_html(row):
    partes = []

    campos = [
        ("discipline", "Disciplina"),
        ("question_type", "Tipo"),
        ("taxonomy_level", "Nível taxonômico"),
        ("base_text", "Texto-base"),
        ("stem", "Enunciado"),
        ("statement_i", "Afirmação I"),
        ("statement_ii", "Afirmação II"),
        ("statement_iii", "Afirmação III"),
        ("statement_iv", "Afirmação IV"),
        ("assertion_i", "Asserção I"),
        ("assertion_ii", "Asserção II"),
    ]

    for col, nome in campos:
        if col in row.index and pd.notna(row[col]) and str(row[col]).strip():
            partes.append(f"<b>{nome}:</b><br/>{clean(row[col])}")

    alts = []
    for col, letra in [
        ("option_a", "A"),
        ("option_b", "B"),
        ("option_c", "C"),
        ("option_d", "D"),
        ("option_e", "E"),
    ]:
        if col in row.index and pd.notna(row[col]) and str(row[col]).strip():
            alts.append(f"<b>{letra})</b> {clean(row[col])}")

    partes.append("<b>Alternativas:</b><br/>" + "<br/>".join(alts))
    partes.append(f"<b>Gabarito:</b> {clean(row['correct_option'])}")

    return "<br/><br/>".join(partes)

def gerar_pdf_comparacao(df_math, df_revisadas, arquivo="comparacao_questoes.pdf"):
    styles = getSampleStyleSheet()

    normal = ParagraphStyle(
        "NormalCustom",
        parent=styles["Normal"],
        fontName="Helvetica",
        fontSize=6.8,
        leading=8.0,
        spaceAfter=2,
    )

    titulo = ParagraphStyle(
        "TituloCustom",
        parent=styles["Heading2"],
        fontName="Helvetica-Bold",
        fontSize=10,
        leading=12,
        spaceAfter=5,
    )

    doc = SimpleDocTemplate(
        arquivo,
        pagesize=A4,
        rightMargin=0.8*cm,
        leftMargin=0.8*cm,
        topMargin=0.8*cm,
        bottomMargin=0.8*cm,
    )

    story = []
    qids = list(df_revisadas["question_id"])

    for qid in qids:
        if qid not in set(df_math["question_id"]):
            continue

        original = df_math[df_math["question_id"] == qid].iloc[0]
        revisada = df_revisadas[df_revisadas["question_id"] == qid].iloc[0]

        original_html = "<b>ORIGINAL</b><br/><br/>" + questao_html(original)
        revisada_html = "<b>REVISADA</b><br/><br/>" + questao_html(revisada)

        story.append(Paragraph(f"Questão {qid}", titulo))

        # tenta colocar lado a lado
        tabela = Table(
            [[
                Paragraph(original_html, normal),
                Paragraph(revisada_html, normal),
            ]],
            colWidths=[9.1*cm, 9.1*cm]
        )

        tabela.setStyle(TableStyle([
            ("VALIGN", (0, 0), (-1, -1), "TOP"),
            ("BOX", (0, 0), (-1, -1), 0.5, colors.black),
            ("INNERGRID", (0, 0), (-1, -1), 0.25, colors.grey),
            ("LEFTPADDING", (0, 0), (-1, -1), 4),
            ("RIGHTPADDING", (0, 0), (-1, -1), 4),
            ("TOPPADDING", (0, 0), (-1, -1), 4),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
        ]))

        try:
            tabela.wrapOn(None, 18.5*cm, 26*cm)
            w, h = tabela.wrap(18.5*cm, 26*cm)

            if h < 24.5*cm:
                story.append(tabela)
                story.append(PageBreak())
            else:
                # se não couber, joga em páginas separadas
                story.append(Paragraph("<b>ORIGINAL</b><br/><br/>" + questao_html(original), normal))
                story.append(PageBreak())
                story.append(Paragraph(f"Questão {qid} — Revisada", titulo))
                story.append(Paragraph("<b>REVISADA</b><br/><br/>" + questao_html(revisada), normal))
                story.append(PageBreak())

        except Exception:
            story.append(Paragraph("<b>ORIGINAL</b><br/><br/>" + questao_html(original), normal))
            story.append(PageBreak())
            story.append(Paragraph(f"Questão {qid} — Revisada", titulo))
            story.append(Paragraph("<b>REVISADA</b><br/><br/>" + questao_html(revisada), normal))
            story.append(PageBreak())

    doc.build(story)
    print(f"PDF gerado: {arquivo}")


In [ ]:
gerar_pdf_comparacao(df_math, df_revisadas, arquivo=str(ARQUIVO_COMPARACAO_PDF))

if files is not None:
    files.download(str(ARQUIVO_COMPARACAO_PDF))


---
# Parte B: geração robusta de novas questões


## 12. Estruturas de dados

Define os modelos Pydantic usados para padronizar questões, revisões, resoluções cegas, rubricas e relatórios de comitê.


In [ ]:
OPCOES = {
    "A": "option_a",
    "B": "option_b",
    "C": "option_c",
    "D": "option_d",
    "E": "option_e",
}


def remover_acentos(texto: Any) -> str:
    return "".join(
        ch
        for ch in unicodedata.normalize("NFD", str(texto or ""))
        if unicodedata.category(ch) != "Mn"
    )


class SympyCheck(BaseModel):
    model_config = ConfigDict(extra="ignore")

    tipo: str = "not_applicable"
    lhs: str = ""
    rhs: str = ""
    expr: str = ""
    expected: str = ""
    variable: str = "x"
    point: str = ""
    direction: str = ""
    rationale: str = ""

    @field_validator("tipo", mode="before")
    @classmethod
    def normalizar_tipo(cls, valor):
        valor = remover_acentos(valor or "not_applicable").strip().lower()
        mapa = {
            "simplificar": "simplify_equals",
            "igualdade": "simplify_equals",
            "derivada": "derivative_equals",
            "integral": "integral_antiderivative",
            "limite": "limit_equals",
            "numerico": "numeric_equals",
            "nao_aplicavel": "not_applicable",
        }
        return mapa.get(valor, valor)


class Questao(BaseModel):
    model_config = ConfigDict(extra="ignore")

    question_id: str = Field(default_factory=lambda: str(uuid.uuid4()))
    discipline: str
    tema: str = ""
    question_type: str = "Multipla Escolha Simples"
    taxonomy_level: str
    base_text: str = ""
    stem: str
    statement_i: str = ""
    statement_ii: str = ""
    statement_iii: str = ""
    statement_iv: str = ""
    assertion_i: str = ""
    assertion_ii: str = ""
    option_a: str
    option_b: str
    option_c: str
    option_d: str
    option_e: str
    correct_option: str
    solution: str = ""
    sympy_checks: List[SympyCheck] = Field(default_factory=list)
    metadata: Dict[str, Any] = Field(default_factory=dict)

    @field_validator("correct_option", mode="before")
    @classmethod
    def normalizar_gabarito(cls, valor):
        texto = remover_acentos(valor).strip().upper()
        texto = texto.replace("OPTION_", "").replace("OPCAO_", "")
        texto = texto.replace("ALTERNATIVA", "").replace("LETRA", "").strip()
        if texto in OPCOES:
            return texto
        match = re.search(r"\b([ABCDE])\b", texto)
        if match:
            return match.group(1)
        return texto


class Problema(BaseModel):
    model_config = ConfigDict(extra="ignore")

    categoria: str = "outro"
    gravidade: str = "media"
    evidencia: str = ""
    correcao_sugerida: str = ""

    @field_validator("gravidade", mode="before")
    @classmethod
    def normalizar_gravidade(cls, valor):
        texto = remover_acentos(valor or "media").strip().lower()
        if texto in {"baixa", "media", "alta", "critica", "critico"}:
            return "critica" if texto in {"critica", "critico"} else texto
        return "media"


class ReviewReport(BaseModel):
    model_config = ConfigDict(extra="ignore")

    fonte: str = ""
    problemas: List[Problema] = Field(default_factory=list)
    resumo: str = ""
    recomendacao: str = ""


class ResolucaoCega(BaseModel):
    model_config = ConfigDict(extra="ignore")

    fonte: str = ""
    alternativa_escolhida: str = ""
    confianca: int = 0
    ha_multiplas_corretas: bool = False
    ha_nenhuma_correta: bool = False
    justificativa: str = ""

    @field_validator("alternativa_escolhida", mode="before")
    @classmethod
    def normalizar_alternativa(cls, valor):
        texto = str(valor or "").strip().upper()
        if texto in OPCOES:
            return texto
        if "MULT" in texto:
            return "MULTIPLAS"
        if "NENH" in texto or "NONE" in texto:
            return "NENHUMA"
        match = re.search(r"\b([ABCDE])\b", texto)
        return match.group(1) if match else texto


class RubricaPedagogica(BaseModel):
    model_config = ConfigDict(extra="ignore")

    correcao_matematica: int = 0
    alternativa_unica_gabarito: int = 0
    clareza_enunciado: int = 0
    qualidade_distratores: int = 0
    adequacao_taxonomica: int = 0
    qualidade_pedagogica: int = 0
    originalidade_diversidade: int = 0
    score_sugerido: int = 0
    problemas_restantes: List[str] = Field(default_factory=list)
    justificativa: str = ""


class CommitteeReport(BaseModel):
    model_config = ConfigDict(extra="ignore")

    concordancias: List[str] = Field(default_factory=list)
    divergencias: List[str] = Field(default_factory=list)
    problemas_corrigidos: List[str] = Field(default_factory=list)
    problemas_pendentes: List[str] = Field(default_factory=list)
    justificativa: str = ""
    questao_revisada: Questao


CAMPOS_AFIRMACOES = [
    "statement_i",
    "statement_ii",
    "statement_iii",
    "statement_iv",
]

CAMPOS_ASSERCOES = [
    "assertion_i",
    "assertion_ii",
]


def _tem_conteudo(valor: Any) -> bool:
    return str(valor or "").strip() != ""


def _peso_campos(obj: Any, campos: List[str]) -> int:
    return sum(len(str(getattr(obj, campo, "") or "").strip()) for campo in campos)


def normalizar_afirmacoes_assercoes_questao(q: Questao) -> Questao:
    tem_afirmacoes = any(_tem_conteudo(getattr(q, campo, "")) for campo in CAMPOS_AFIRMACOES)
    tem_assercoes = any(_tem_conteudo(getattr(q, campo, "")) for campo in CAMPOS_ASSERCOES)

    if tem_afirmacoes and tem_assercoes:
        tipo = remover_acentos(q.question_type).lower()
        manter_assercoes = "asserc" in tipo or _peso_campos(q, CAMPOS_ASSERCOES) > _peso_campos(q, CAMPOS_AFIRMACOES)
        limpar = CAMPOS_AFIRMACOES if manter_assercoes else CAMPOS_ASSERCOES
        for campo in limpar:
            setattr(q, campo, "")

    return q


def normalizar_afirmacoes_assercoes_dict(q: dict) -> dict:
    q = dict(q or {})
    tem_afirmacoes = any(_tem_conteudo(q.get(campo, "")) for campo in CAMPOS_AFIRMACOES)
    tem_assercoes = any(_tem_conteudo(q.get(campo, "")) for campo in CAMPOS_ASSERCOES)

    if tem_afirmacoes and tem_assercoes:
        tipo = remover_acentos(q.get("question_type", "")).lower()
        peso_afirmacoes = sum(len(str(q.get(campo, "") or "").strip()) for campo in CAMPOS_AFIRMACOES)
        peso_assercoes = sum(len(str(q.get(campo, "") or "").strip()) for campo in CAMPOS_ASSERCOES)
        manter_assercoes = "asserc" in tipo or peso_assercoes > peso_afirmacoes
        limpar = CAMPOS_AFIRMACOES if manter_assercoes else CAMPOS_ASSERCOES
        for campo in limpar:
            q[campo] = ""

    return q


## 13. Utilidades de JSON e texto

Funções auxiliares para extrair JSON, normalizar respostas dos modelos e converter uma questão estruturada em texto.


In [ ]:
def extrair_json(texto: Optional[str]) -> Optional[dict]:
    if texto is None:
        return None

    texto = texto.strip()
    texto = re.sub(r"^```json", "", texto, flags=re.IGNORECASE).strip()
    texto = re.sub(r"^```", "", texto).strip()
    texto = re.sub(r"```$", "", texto).strip()

    try:
        return json.loads(texto)
    except Exception:
        pass

    match = re.search(r"\{.*\}", texto, flags=re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            return None

    return None


def dump(obj: Any) -> Any:
    if isinstance(obj, BaseModel):
        return obj.model_dump()
    if isinstance(obj, list):
        return [dump(x) for x in obj]
    if isinstance(obj, dict):
        return {k: dump(v) for k, v in obj.items()}
    if isinstance(obj, Path):
        return str(obj)
    return obj


def parse_questao(obj: Any) -> Questao:
    if isinstance(obj, Questao):
        return normalizar_afirmacoes_assercoes_questao(obj)
    if isinstance(obj, dict) and "questao" in obj:
        obj = obj["questao"]
    if isinstance(obj, dict) and "questao_revisada" in obj:
        obj = obj["questao_revisada"]
    return normalizar_afirmacoes_assercoes_questao(Questao.model_validate(obj))


def parse_review(obj: Optional[dict], fonte: str) -> ReviewReport:
    if obj is None:
        return ReviewReport(
            fonte=fonte,
            problemas=[
                Problema(
                    categoria="falha_chamada_modelo",
                    gravidade="alta",
                    evidencia="O modelo nao retornou JSON valido.",
                    correcao_sugerida="Reexecutar a avaliacao.",
                )
            ],
            resumo="Falha ao obter revisao.",
            recomendacao="Reexecutar.",
        )

    problemas = obj.get("problemas", [])
    normalizados = []
    for problema in problemas:
        if isinstance(problema, str):
            normalizados.append(
                {
                    "categoria": "outro",
                    "gravidade": "media",
                    "evidencia": problema,
                    "correcao_sugerida": "",
                }
            )
        elif isinstance(problema, dict):
            normalizados.append(problema)

    obj = dict(obj)
    obj["fonte"] = fonte
    obj["problemas"] = normalizados
    return ReviewReport.model_validate(obj)


def clamp_score(valor: Any) -> int:
    try:
        return int(max(0, min(100, round(float(valor)))))
    except Exception:
        return 0


In [ ]:
def questao_para_texto(
    q: Questao,
    incluir_gabarito: bool = True,
    incluir_solucao: bool = False,
    incluir_checks: bool = False,
) -> str:
    partes = [
        f"ID da questao: {q.question_id}",
        f"Disciplina: {q.discipline}",
        f"Tema: {q.tema}",
        f"Tipo de questao: {q.question_type}",
        f"Nivel taxonomico declarado: {q.taxonomy_level}",
    ]

    if q.base_text.strip():
        partes.append(f"\nTexto-base:\n{q.base_text}")

    partes.append(f"\nEnunciado:\n{q.stem}")

    for nome, label in [
        ("statement_i", "I"),
        ("statement_ii", "II"),
        ("statement_iii", "III"),
        ("statement_iv", "IV"),
        ("assertion_i", "Assercao I"),
        ("assertion_ii", "Assercao II"),
    ]:
        valor = getattr(q, nome)
        if str(valor).strip():
            partes.append(f"\n{label}:\n{valor}")

    partes.append("\nAlternativas:")
    for letra, campo in OPCOES.items():
        partes.append(f"{letra}) {getattr(q, campo)}")

    if incluir_gabarito:
        partes.append(f"\nGabarito informado: {q.correct_option}")

    if incluir_solucao and q.solution.strip():
        partes.append(f"\nSolucao esperada:\n{q.solution}")

    if incluir_checks and q.sympy_checks:
        partes.append("\nChecagens SymPy propostas:")
        partes.append(json.dumps([c.model_dump() for c in q.sympy_checks], ensure_ascii=False, indent=2))

    return "\n".join(partes)


def resumo_problemas(revisoes: List[ReviewReport]) -> str:
    linhas = []
    for revisao in revisoes:
        linhas.append(f"\nFonte: {revisao.fonte}")
        if not revisao.problemas:
            linhas.append("- Sem problemas apontados.")
            continue
        for p in revisao.problemas:
            linhas.append(f"- [{p.gravidade}] {p.categoria}: {p.evidencia} | Sugestao: {p.correcao_sugerida}")
    return "\n".join(linhas)


## 14. Chamadas aos modelos

Camadas de chamada para OpenAI e Sabiá, com tentativas automáticas e extração de JSON.


In [ ]:
def chamada_openai_json(
    model: str,
    system: str,
    user: str,
    temperature: Optional[float] = 0.0,
    max_retries: int = 3,
) -> tuple[Optional[dict], str]:
    bruto = ""
    ultimo_erro = None
    usar_temperature = temperature is not None
    usar_response_format = True

    for tentativa in range(1, max_retries + 1):
        try:
            kwargs = {
                "model": model,
                "messages": [
                    {"role": "system", "content": system},
                    {"role": "user", "content": user},
                ],
            }

            if usar_response_format:
                kwargs["response_format"] = {"type": "json_object"}
            if usar_temperature:
                kwargs["temperature"] = temperature

            resp = client_openai.chat.completions.create(**kwargs)
            bruto = resp.choices[0].message.content or ""
            obj = extrair_json(bruto)

            if obj is not None:
                return obj, bruto

            ultimo_erro = ValueError("JSON invalido ou ausente.")

        except Exception as e:
            ultimo_erro = e
            msg = str(e).lower()
            if "temperature" in msg:
                usar_temperature = False
            if "response_format" in msg or "json_object" in msg:
                usar_response_format = False

        time.sleep(2 * tentativa)

    print("Falha em chamada OpenAI:", ultimo_erro)
    if bruto:
        print(bruto[:1500])
    return None, bruto


def chamada_sabia_json(
    model: str,
    system: str,
    user: str,
    temperature: float = 0.0,
    max_retries: int = 3,
) -> tuple[Optional[dict], str]:
    if client_sabia is None:
        return None, ""

    bruto = ""
    ultimo_erro = None

    for tentativa in range(1, max_retries + 1):
        try:
            resp = client_sabia.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": user},
                ],
                temperature=temperature,
                max_tokens=2500,
            )
            bruto = resp.choices[0].message.content or ""
            obj = extrair_json(bruto)
            if obj is not None:
                return obj, bruto
            ultimo_erro = ValueError("JSON invalido ou ausente.")
        except Exception as e:
            ultimo_erro = e

        time.sleep(2 * tentativa)

    print("Falha em chamada Sabia:", ultimo_erro)
    if bruto:
        print(bruto[:1500])
    return None, bruto


def chamada_modelo_json(
    provedor: str,
    model: str,
    system: str,
    user: str,
    temperature: Optional[float] = 0.0,
    max_retries: int = 3,
) -> tuple[Optional[dict], str]:
    provedor = str(provedor or "openai").lower().strip()

    if provedor == "sabia":
        if client_sabia is not None:
            return chamada_sabia_json(
                model,
                system,
                user,
                temperature=0.0 if temperature is None else float(temperature),
                max_retries=max_retries,
            )

        print("client_sabia nao definido; usando OpenAI como fallback.")

    return chamada_openai_json(
        model if provedor == "openai" else MODEL_REVISOR_OPENAI,
        system,
        user,
        temperature=temperature,
        max_retries=max_retries,
    )


## 15. Validadores objetivos

O validador estrutural verifica campos obrigatórios, gabarito, alternativas duplicadas e sinais formais. O validador simbólico usa SymPy quando a questão trouxer checagens executáveis.


In [ ]:
def texto_normalizado(texto: str) -> str:
    texto = str(texto or "").strip().lower()
    texto = re.sub(r"\s+", " ", texto)
    texto = re.sub(r"[^\w\s+\-*/^=.,;()]", "", texto)
    return texto


def validar_estrutura(q: Questao) -> dict:
    problemas = []
    penalidade = 0

    obrigatorios = [
        "discipline",
        "question_type",
        "taxonomy_level",
        "stem",
        "option_a",
        "option_b",
        "option_c",
        "option_d",
        "option_e",
        "correct_option",
    ]

    for campo in obrigatorios:
        valor = str(getattr(q, campo, "") or "").strip()
        if not valor:
            problemas.append(
                {
                    "tipo": "campo_obrigatorio_vazio",
                    "gravidade": "alta",
                    "campo": campo,
                    "mensagem": f"Campo obrigatorio vazio: {campo}",
                }
            )
            penalidade += 20

    if q.correct_option not in OPCOES:
        problemas.append(
            {
                "tipo": "gabarito_invalido",
                "gravidade": "critica",
                "campo": "correct_option",
                "mensagem": "Gabarito deve ser uma letra entre A e E.",
            }
        )
        penalidade += 40

    alternativas = [getattr(q, campo) for campo in OPCOES.values()]
    alternativas_norm = [texto_normalizado(a) for a in alternativas]
    duplicadas = sorted({a for a in alternativas_norm if alternativas_norm.count(a) > 1 and a})
    if duplicadas:
        problemas.append(
            {
                "tipo": "alternativas_duplicadas",
                "gravidade": "alta",
                "mensagem": "Ha alternativas duplicadas ou praticamente identicas.",
                "evidencia": duplicadas,
            }
        )
        penalidade += 25

    proibidas = [
        "todas as alternativas",
        "nenhuma das alternativas",
        "todas estao corretas",
        "nenhuma esta correta",
    ]
    for letra, campo in OPCOES.items():
        texto = texto_normalizado(getattr(q, campo))
        if any(expr in texto for expr in proibidas):
            problemas.append(
                {
                    "tipo": "alternativa_meta",
                    "gravidade": "media",
                    "campo": campo,
                    "mensagem": f"Alternativa {letra} usa formulacao meta como 'todas/nenhuma'.",
                }
            )
            penalidade += 10

    if len(q.stem.strip()) < 40:
        problemas.append(
            {
                "tipo": "enunciado_curto",
                "gravidade": "media",
                "campo": "stem",
                "mensagem": "Enunciado parece curto demais para uma questao de ensino superior.",
            }
        )
        penalidade += 8

    if not q.solution.strip():
        problemas.append(
            {
                "tipo": "solucao_ausente",
                "gravidade": "media",
                "campo": "solution",
                "mensagem": "A questao nao trouxe solucao esperada, dificultando auditoria.",
            }
        )
        penalidade += 8

    return {
        "ok": not any(p["gravidade"] in {"alta", "critica"} for p in problemas),
        "problemas": problemas,
        "penalidade": min(70, penalidade),
    }


In [ ]:
TRANSFORMACOES = standard_transformations + (
    implicit_multiplication_application,
    convert_xor,
)

SAFE_SYMBOLS = {nome: sp.symbols(nome) for nome in list("abcdefghijklmnopqrstuvwxyz")}
SAFE_FUNCS = {
    "sin": sp.sin,
    "cos": sp.cos,
    "tan": sp.tan,
    "asin": sp.asin,
    "acos": sp.acos,
    "atan": sp.atan,
    "exp": sp.exp,
    "log": sp.log,
    "ln": sp.log,
    "sqrt": sp.sqrt,
    "pi": sp.pi,
    "E": sp.E,
    "Abs": sp.Abs,
}
SAFE_LOCAL_DICT = {**SAFE_SYMBOLS, **SAFE_FUNCS}


def parse_sympy_seguro(expr: str):
    expr = str(expr or "").strip()
    if not expr:
        raise ValueError("expressao vazia")

    proibidos = ["__", "import", "lambda", "exec", "eval", ";", "'", '"', "`"]
    if any(tok in expr for tok in proibidos):
        raise ValueError("expressao contem token proibido")

    if len(expr) > 300:
        raise ValueError("expressao longa demais")

    return parse_expr(
        expr.replace("^", "**"),
        local_dict=SAFE_LOCAL_DICT,
        transformations=TRANSFORMACOES,
        evaluate=True,
    )


def validar_sympy(q: Questao) -> dict:
    resultados = []
    falhas = 0
    erros_execucao = 0
    executados = 0
    ignorados = 0

    for check in q.sympy_checks or []:
        tipo = check.tipo

        if tipo == "not_applicable":
            ignorados += 1
            resultados.append(
                {
                    "tipo": tipo,
                    "status": "ignorado",
                    "rationale": check.rationale,
                }
            )
            continue

        try:
            executados += 1

            if tipo == "simplify_equals":
                lhs = parse_sympy_seguro(check.lhs)
                rhs = parse_sympy_seguro(check.rhs)
                ok = bool(sp.simplify(lhs - rhs) == 0)

            elif tipo == "numeric_equals":
                lhs = parse_sympy_seguro(check.lhs)
                rhs = parse_sympy_seguro(check.rhs)
                ok = abs(float(sp.N(lhs - rhs))) < 1e-8

            elif tipo == "derivative_equals":
                var = SAFE_SYMBOLS.get(check.variable, sp.symbols(check.variable))
                expr = parse_sympy_seguro(check.expr)
                expected = parse_sympy_seguro(check.expected)
                ok = bool(sp.simplify(sp.diff(expr, var) - expected) == 0)

            elif tipo == "integral_antiderivative":
                var = SAFE_SYMBOLS.get(check.variable, sp.symbols(check.variable))
                expr = parse_sympy_seguro(check.expr)
                expected = parse_sympy_seguro(check.expected)
                ok = bool(sp.simplify(sp.diff(expected, var) - expr) == 0)

            elif tipo == "limit_equals":
                var = SAFE_SYMBOLS.get(check.variable, sp.symbols(check.variable))
                expr = parse_sympy_seguro(check.expr)
                point = parse_sympy_seguro(check.point)
                expected = parse_sympy_seguro(check.expected)
                direction = check.direction.strip() or "+"
                if direction not in {"+", "-", "+-"}:
                    direction = "+"
                value = sp.limit(expr, var, point, dir=direction)
                ok = bool(sp.simplify(value - expected) == 0)

            else:
                ignorados += 1
                resultados.append(
                    {
                        "tipo": tipo,
                        "status": "ignorado",
                        "mensagem": "Tipo de check nao reconhecido.",
                    }
                )
                continue

            if not ok:
                falhas += 1
            resultados.append({"tipo": tipo, "status": "ok" if ok else "falhou"})

        except Exception as e:
            erros_execucao += 1
            resultados.append(
                {
                    "tipo": tipo,
                    "status": "erro_execucao",
                    "erro": str(e),
                }
            )

    if executados == 0:
        return {
            "status": "nao_verificado",
            "ok": True,
            "penalidade": 4 if ignorados == 0 else 0,
            "resultados": resultados,
            "mensagem": "Nenhuma checagem simbolica executavel foi fornecida.",
        }

    penalidade = falhas * 30 + erros_execucao * 8
    return {
        "status": "ok" if falhas == 0 and erros_execucao == 0 else "falhou",
        "ok": falhas == 0,
        "penalidade": min(60, penalidade),
        "resultados": resultados,
        "executados": executados,
        "falhas": falhas,
        "erros_execucao": erros_execucao,
    }


## 16. Autor, revisores, resolvedores e auditoria

Este bloco implementa os papéis do pipeline robusto:

- autor;
- revisor GPT;
- revisor Sabiá;
- resolvedores cegos;
- auditor por rubrica;
- comitê de reescrita.


In [ ]:
TEMPLATE_QUESTAO_JSON = """
{
  "question_id": "",
  "discipline": "",
  "tema": "",
  "question_type": "Multipla Escolha Simples",
  "taxonomy_level": "Aplicar | Analisar | Avaliar",
  "base_text": "",
  "stem": "",
  "statement_i": "",
  "statement_ii": "",
  "statement_iii": "",
  "statement_iv": "",
  "assertion_i": "",
  "assertion_ii": "",
  "option_a": "",
  "option_b": "",
  "option_c": "",
  "option_d": "",
  "option_e": "",
  "correct_option": "A|B|C|D|E",
  "solution": "resolucao esperada, suficiente para auditar o gabarito",
  "sympy_checks": [
    {
      "tipo": "simplify_equals | derivative_equals | integral_antiderivative | limit_equals | numeric_equals | not_applicable",
      "lhs": "",
      "rhs": "",
      "expr": "",
      "expected": "",
      "variable": "x",
      "point": "",
      "direction": "",
      "rationale": ""
    }
  ]
}
"""


SYSTEM_ESPECIALISTA = (
    "Voce e especialista em avaliacao educacional e elaboracao de questoes "
    "de Matematica para o Ensino Superior. Responda sempre em JSON valido, "
    "sem markdown e sem texto fora do JSON."
)


def gerar_questao(item: dict) -> tuple[Questao, str]:
    prompt = f"""
Elabore UMA questao de Matematica de alta qualidade para Ensino Superior.

Dados obrigatorios:
- Disciplina: {item["discipline"]}
- Tema: {item["tema"]}
- Tipo: {item.get("question_type", "Multipla Escolha Simples")}
- Nivel taxonomico: {item["taxonomy_level"]}

Criterios obrigatorios:
- A questao deve atingir de fato o nivel taxonomico indicado.
- Deve haver apenas uma alternativa correta.
- Os distratores devem ser plausiveis e representar erros reais de estudantes.
- Evite alternativas absurdas, "todas as alternativas" e "nenhuma das alternativas".
- Evite texto-base decorativo.
- Use texto-base somente se ele for necessario.
- Voce pode usar afirmacoes OU assercoes quando o tipo da questao exigir, mas nunca os dois formatos na mesma questao.
- Se usar afirmacoes, deixe assertion_i e assertion_ii vazios.
- Se usar assercoes, deixe statement_i, statement_ii, statement_iii e statement_iv vazios.
- Para Multipla Escolha Simples comum, prefira enunciado direto sem afirmacoes nem assercoes.
- Use notacao matematica clara.
- Inclua solucao esperada para auditoria.
- Inclua checagens SymPy quando forem aplicaveis. Use sintaxe SymPy simples, nao LaTeX.
- Se SymPy nao se aplicar, use um unico check com tipo "not_applicable" e explique em rationale.

Retorne SOMENTE JSON neste formato:

{TEMPLATE_QUESTAO_JSON}
"""

    obj, bruto = chamada_modelo_json(
        PROVEDOR_AUTOR,
        MODEL_AUTOR,
        SYSTEM_ESPECIALISTA,
        prompt,
        temperature=0.4,
        max_retries=3,
    )
    if obj is None:
        raise ValueError("Autor nao retornou JSON valido.")

    q = parse_questao(obj)
    q.tema = q.tema or item["tema"]
    q.discipline = q.discipline or item["discipline"]
    q.question_type = q.question_type or item.get("question_type", "Multipla Escolha Simples")
    q.taxonomy_level = q.taxonomy_level or item["taxonomy_level"]
    q.metadata["plano"] = item
    q = normalizar_afirmacoes_assercoes_questao(q)
    return q, bruto


In [ ]:
def revisar_problemas_modelo(q: Questao, fonte: str = "openai") -> tuple[ReviewReport, str]:
    texto = questao_para_texto(q, incluir_gabarito=True, incluir_solucao=True, incluir_checks=True)

    prompt = f"""
Avalie rigorosamente a questao abaixo.

Procure todos os problemas pedagogicos, conceituais, formais e matematicos.
Considere:
- erro conceitual ou de calculo;
- gabarito incorreto;
- mais de uma alternativa correta;
- ausencia de alternativa correta;
- enunciado ambiguo;
- texto-base irrelevante;
- distratores fracos;
- alternativa com pista formal;
- incompatibilidade com o nivel taxonomico;
- notacao inadequada;
- solucao esperada inconsistente;
- checagens simbolicas ausentes ou ruins.

Retorne SOMENTE JSON neste formato:

{{
  "problemas": [
    {{
      "categoria": "",
      "gravidade": "baixa | media | alta | critica",
      "evidencia": "",
      "correcao_sugerida": ""
    }}
  ],
  "resumo": "",
  "recomendacao": ""
}}

Se a questao estiver adequada, retorne "problemas": [].

QUESTAO:
{texto}
"""

    if fonte == "sabia" and client_sabia is not None:
        obj, bruto = chamada_sabia_json(MODEL_SABIA, SYSTEM_ESPECIALISTA, prompt, temperature=0.0)
        return parse_review(obj, "sabia-4"), bruto

    if fonte == "sabia" and client_sabia is None:
        system = (
            SYSTEM_ESPECIALISTA
            + " Voce deve atuar como revisor B, cetico e independente do autor."
        )
        obj, bruto = chamada_modelo_json(
            PROVEDOR_REVISOR_OPENAI,
            MODEL_REVISOR_OPENAI,
            system,
            prompt,
            temperature=0.0,
            max_retries=3,
        )
        return parse_review(obj, "openai_revisor_b_fallback"), bruto

    obj, bruto = chamada_modelo_json(
        PROVEDOR_REVISOR_OPENAI,
        MODEL_REVISOR_OPENAI,
        SYSTEM_ESPECIALISTA,
        prompt,
        temperature=0.0,
        max_retries=3,
    )
    return parse_review(obj, "openai_revisor_a"), bruto


def resolver_sem_gabarito(q: Questao, fonte: str = "openai") -> tuple[ResolucaoCega, str]:
    texto = questao_para_texto(q, incluir_gabarito=False, incluir_solucao=False, incluir_checks=False)

    prompt = f"""
Resolva a questao abaixo sem acesso ao gabarito.

Retorne SOMENTE JSON:

{{
  "alternativa_escolhida": "A | B | C | D | E | MULTIPLAS | NENHUMA",
  "confianca": 0,
  "ha_multiplas_corretas": false,
  "ha_nenhuma_correta": false,
  "justificativa": ""
}}

QUESTAO:
{texto}
"""

    if fonte == "sabia" and client_sabia is not None:
        obj, bruto = chamada_sabia_json(MODEL_SABIA, SYSTEM_ESPECIALISTA, prompt, temperature=0.0)
        fonte_nome = "sabia-4_resolvedor"
    else:
        obj, bruto = chamada_modelo_json(
            PROVEDOR_RESOLVEDOR,
            MODEL_RESOLVEDOR,
            SYSTEM_ESPECIALISTA,
            prompt,
            temperature=0.0,
            max_retries=3,
        )
        fonte_nome = "openai_resolvedor" if fonte == "openai" else "openai_resolvedor_b_fallback"

    if obj is None:
        return (
            ResolucaoCega(
                fonte=fonte_nome,
                alternativa_escolhida="",
                confianca=0,
                justificativa="Falha ao obter resolucao cega.",
            ),
            bruto,
        )

    obj = dict(obj)
    obj["fonte"] = fonte_nome
    return ResolucaoCega.model_validate(obj), bruto


In [ ]:
def avaliar_rubrica(
    q: Questao,
    validacao_estrutura: dict,
    validacao_sympy: dict,
    revisoes: List[ReviewReport],
    resolucoes: List[ResolucaoCega],
) -> tuple[RubricaPedagogica, str]:
    contexto = {
        "validacao_estrutural": validacao_estrutura,
        "validacao_sympy": validacao_sympy,
        "revisoes": [r.model_dump() for r in revisoes],
        "resolucoes_cegas": [r.model_dump() for r in resolucoes],
    }

    prompt = f"""
Avalie a qualidade pedagogica da questao usando a rubrica abaixo.

De notas de 0 a 100 para:
- correcao_matematica;
- alternativa_unica_gabarito;
- clareza_enunciado;
- qualidade_distratores;
- adequacao_taxonomica;
- qualidade_pedagogica;
- originalidade_diversidade.

Use as validacoes e revisoes como evidencias. Nao ignore falhas estruturais,
falhas SymPy ou discordancia dos resolvedores cegos.

Retorne SOMENTE JSON:

{{
  "correcao_matematica": 0,
  "alternativa_unica_gabarito": 0,
  "clareza_enunciado": 0,
  "qualidade_distratores": 0,
  "adequacao_taxonomica": 0,
  "qualidade_pedagogica": 0,
  "originalidade_diversidade": 0,
  "score_sugerido": 0,
  "problemas_restantes": [],
  "justificativa": ""
}}

QUESTAO:
{questao_para_texto(q, incluir_gabarito=True, incluir_solucao=True, incluir_checks=True)}

EVIDENCIAS:
{json.dumps(contexto, ensure_ascii=False, indent=2)}
"""

    obj, bruto = chamada_modelo_json(
        PROVEDOR_AUDITOR,
        MODEL_AUDITOR,
        SYSTEM_ESPECIALISTA,
        prompt,
        temperature=0.0,
        max_retries=3,
    )

    if obj is None:
        rubrica = RubricaPedagogica(
            problemas_restantes=["Auditor nao retornou JSON valido."],
            justificativa="Falha na auditoria.",
        )
    else:
        rubrica = RubricaPedagogica.model_validate(obj)

    for campo in [
        "correcao_matematica",
        "alternativa_unica_gabarito",
        "clareza_enunciado",
        "qualidade_distratores",
        "adequacao_taxonomica",
        "qualidade_pedagogica",
        "originalidade_diversidade",
        "score_sugerido",
    ]:
        setattr(rubrica, campo, clamp_score(getattr(rubrica, campo)))

    return rubrica, bruto


def calcular_score_final(
    q: Questao,
    rubrica: RubricaPedagogica,
    validacao_estrutura: dict,
    validacao_sympy: dict,
    revisoes: List[ReviewReport],
    resolucoes: List[ResolucaoCega],
) -> dict:
    pesos = {
        "correcao_matematica": 0.30,
        "alternativa_unica_gabarito": 0.20,
        "clareza_enunciado": 0.12,
        "qualidade_distratores": 0.14,
        "adequacao_taxonomica": 0.12,
        "qualidade_pedagogica": 0.08,
        "originalidade_diversidade": 0.04,
    }

    score_rubrica = sum(getattr(rubrica, k) * peso for k, peso in pesos.items())
    penalidade = 0

    penalidade += validacao_estrutura.get("penalidade", 0)
    penalidade += validacao_sympy.get("penalidade", 0)

    problemas_altos = 0
    problemas_criticos = 0
    for revisao in revisoes:
        for problema in revisao.problemas:
            if problema.gravidade == "critica":
                problemas_criticos += 1
            elif problema.gravidade == "alta":
                problemas_altos += 1

    penalidade += min(35, problemas_criticos * 18 + problemas_altos * 10)

    for resolucao in resolucoes:
        alternativa = resolucao.alternativa_escolhida
        if resolucao.ha_multiplas_corretas or alternativa == "MULTIPLAS":
            penalidade += 30
        elif resolucao.ha_nenhuma_correta or alternativa == "NENHUMA":
            penalidade += 30
        elif alternativa in OPCOES and alternativa != q.correct_option and resolucao.confianca >= 60:
            penalidade += 22

    if rubrica.problemas_restantes:
        penalidade += min(20, 5 * len(rubrica.problemas_restantes))

    score_final = max(0, min(score_rubrica, 100 - penalidade))

    aprovado = (
        score_final >= LIMIAR_APROVACAO
        and validacao_estrutura.get("ok", False)
        and validacao_sympy.get("ok", True)
        and problemas_criticos == 0
        and problemas_altos == 0
    )

    return {
        "score_rubrica": round(score_rubrica, 2),
        "penalidade": round(penalidade, 2),
        "score_final": round(score_final, 2),
        "status": "APROVADA" if aprovado else "REPROVADA",
        "problemas_altos": problemas_altos,
        "problemas_criticos": problemas_criticos,
    }


In [ ]:
def comite_revisao(
    q: Questao,
    item_plano: dict,
    validacao_estrutura: dict,
    validacao_sympy: dict,
    revisoes: List[ReviewReport],
    resolucoes: List[ResolucaoCega],
    rubrica: RubricaPedagogica,
    score: dict,
    auditoria_anterior: Optional[dict] = None,
) -> tuple[CommitteeReport, str]:
    contexto = {
        "plano": item_plano,
        "validacao_estrutural": validacao_estrutura,
        "validacao_sympy": validacao_sympy,
        "revisoes": [r.model_dump() for r in revisoes],
        "resolucoes_cegas": [r.model_dump() for r in resolucoes],
        "rubrica": rubrica.model_dump(),
        "score": score,
        "auditoria_anterior": auditoria_anterior,
    }

    prompt = f"""
Atue como comite de revisao de questoes de Matematica.

Sua tarefa:
1. Identificar concordancias entre revisores, validadores e resolvedores.
2. Identificar divergencias.
3. Decidir quais problemas realmente precisam ser corrigidos.
4. Reescrever completamente a questao, preservando disciplina, tema, tipo e objetivo.
5. Garantir apenas uma alternativa correta.
6. Melhorar distratores.
7. Corrigir linguagem, notacao matematica, ambiguidades, gabarito e solucao.
8. Atualizar as checagens SymPy. Use sintaxe SymPy simples quando aplicavel.
9. Pode usar afirmacoes OU assercoes quando o formato da questao exigir, mas nunca os dois formatos na mesma questao.
10. Se usar afirmacoes, deixe assertion_i e assertion_ii vazios.
11. Se usar assercoes, deixe statement_i, statement_ii, statement_iii e statement_iv vazios.
12. Para Multipla Escolha Simples comum, prefira enunciado direto sem afirmacoes nem assercoes.

Nao esconda problemas: se algo nao puder ser validado por SymPy, use not_applicable com rationale claro.

Retorne SOMENTE JSON neste formato:

{{
  "concordancias": [],
  "divergencias": [],
  "problemas_corrigidos": [],
  "problemas_pendentes": [],
  "justificativa": "",
  "questao_revisada": {TEMPLATE_QUESTAO_JSON}
}}

QUESTAO ATUAL:
{questao_para_texto(q, incluir_gabarito=True, incluir_solucao=True, incluir_checks=True)}

EVIDENCIAS:
{json.dumps(contexto, ensure_ascii=False, indent=2)}
"""

    obj, bruto = chamada_modelo_json(
        PROVEDOR_COMITE,
        MODEL_COMITE,
        SYSTEM_ESPECIALISTA,
        prompt,
        temperature=0.2,
        max_retries=3,
    )

    if obj is None:
        raise ValueError("Comite nao retornou JSON valido.")

    report = CommitteeReport.model_validate(obj)
    report.questao_revisada.tema = report.questao_revisada.tema or item_plano["tema"]
    report.questao_revisada.discipline = report.questao_revisada.discipline or item_plano["discipline"]
    report.questao_revisada.taxonomy_level = report.questao_revisada.taxonomy_level or item_plano["taxonomy_level"]
    report.questao_revisada.question_type = report.questao_revisada.question_type or item_plano.get(
        "question_type", "Multipla Escolha Simples"
    )
    report.questao_revisada.metadata["plano"] = item_plano
    report.questao_revisada = normalizar_afirmacoes_assercoes_questao(report.questao_revisada)
    return report, bruto


## 17. Plano de geração

O plano combina disciplinas, temas e níveis taxonômicos. Para teste rápido, ele usa `N_QUESTOES = 2`; para o plano completo, altere esse valor para `60` na configuração inicial.


In [ ]:
def criar_plano_padrao(n: int = N_QUESTOES) -> List[dict]:
    disciplinas_temas = {
        "Geometria Espacial": [
            "posicao relativa entre retas e planos",
            "distancia entre ponto, reta e plano",
            "areas e volumes de solidos",
            "poliedros e relacao de Euler",
            "secoes planas em solidos",
        ],
        "Calculo Diferencial e Integral III": [
            "gradiente e derivada direcional",
            "plano tangente",
            "integrais duplas",
            "integrais triplas",
            "mudanca de variaveis e jacobiano",
        ],
        "Calculo Diferencial e Integral II": [
            "sequencias e series",
            "series de Taylor",
            "integrais improprias",
            "tecnicas de integracao",
        ],
        "Estruturas Algebricas": [
            "relacoes e operacoes",
            "grupos",
            "subgrupos",
            "homomorfismos",
            "conjuntos e operacoes",
        ],
    }

    niveis = ["Aplicar", "Analisar", "Avaliar"]
    plano = []
    for disciplina, temas in disciplinas_temas.items():
        for tema in temas:
            for nivel in niveis:
                plano.append(
                    {
                        "discipline": disciplina,
                        "tema": tema,
                        "taxonomy_level": nivel,
                        "question_type": "Multipla Escolha Simples",
                    }
                )

    return plano[:n]


def carregar_plano_csv(caminho: str) -> List[dict]:
    df = pd.read_csv(caminho)
    colunas = {"discipline", "tema", "taxonomy_level"}
    faltantes = colunas - set(df.columns)
    if faltantes:
        raise ValueError(f"CSV do plano precisa das colunas: {sorted(colunas)}. Faltam: {sorted(faltantes)}")

    if "question_type" not in df.columns:
        df["question_type"] = "Multipla Escolha Simples"

    return df[["discipline", "tema", "taxonomy_level", "question_type"]].to_dict("records")


# Para usar um CSV proprio, envie o arquivo ao Colab e troque por:
# plano_questoes = carregar_plano_csv("/content/seu_plano.csv")
plano_questoes = criar_plano_padrao(N_QUESTOES)
pd.DataFrame(plano_questoes).head()


## 18. Execução do pipeline robusto

Cada questão passa por geração, revisão, validação, auditoria, reescrita e escolha da melhor versão.


In [ ]:
def salvar_resultados(resultados: List[dict]):
    with open(ARQ_JSON, "w", encoding="utf-8") as f:
        json.dump(dump(resultados), f, ensure_ascii=False, indent=2)

    linhas = []
    for r in resultados:
        q = r.get("questao_final") or {}
        aud = r.get("auditoria_final") or {}
        score = aud.get("score", {}) if isinstance(aud, dict) else {}
        rubrica = aud.get("rubrica", {}) if isinstance(aud, dict) else {}

        linhas.append(
            {
                "numero": r.get("numero"),
                "status_final": r.get("status_final", ""),
                "score_final": r.get("score_final", ""),
                "n_revisoes": r.get("n_revisoes", ""),
                "discipline": q.get("discipline", ""),
                "tema": q.get("tema", r.get("plano", {}).get("tema", "")),
                "question_type": q.get("question_type", ""),
                "taxonomy_level": q.get("taxonomy_level", ""),
                "base_text": q.get("base_text", ""),
                "stem": q.get("stem", ""),
                "statement_i": q.get("statement_i", ""),
                "statement_ii": q.get("statement_ii", ""),
                "statement_iii": q.get("statement_iii", ""),
                "statement_iv": q.get("statement_iv", ""),
                "assertion_i": q.get("assertion_i", ""),
                "assertion_ii": q.get("assertion_ii", ""),
                "option_a": q.get("option_a", ""),
                "option_b": q.get("option_b", ""),
                "option_c": q.get("option_c", ""),
                "option_d": q.get("option_d", ""),
                "option_e": q.get("option_e", ""),
                "correct_option": q.get("correct_option", ""),
                "solution": q.get("solution", ""),
                "score_rubrica": score.get("score_rubrica", ""),
                "penalidade": score.get("penalidade", ""),
                "correcao_matematica": rubrica.get("correcao_matematica", ""),
                "alternativa_unica_gabarito": rubrica.get("alternativa_unica_gabarito", ""),
                "clareza_enunciado": rubrica.get("clareza_enunciado", ""),
                "qualidade_distratores": rubrica.get("qualidade_distratores", ""),
                "adequacao_taxonomica": rubrica.get("adequacao_taxonomica", ""),
                "qualidade_pedagogica": rubrica.get("qualidade_pedagogica", ""),
                "originalidade_diversidade": rubrica.get("originalidade_diversidade", ""),
            }
        )

    pd.DataFrame(linhas).to_csv(ARQ_CSV, index=False)


def auditar_iteracao(q: Questao):
    validacao_estrutura = validar_estrutura(q)
    validacao_sympy = validar_sympy(q)

    revisao_openai, bruto_rev_openai = revisar_problemas_modelo(q, fonte="openai")
    time.sleep(SLEEP_ENTRE_CHAMADAS)

    revisao_sabia, bruto_rev_sabia = revisar_problemas_modelo(q, fonte="sabia")
    time.sleep(SLEEP_ENTRE_CHAMADAS)

    resolucao_openai, bruto_res_openai = resolver_sem_gabarito(q, fonte="openai")
    time.sleep(SLEEP_ENTRE_CHAMADAS)

    resolucao_sabia, bruto_res_sabia = resolver_sem_gabarito(q, fonte="sabia")
    time.sleep(SLEEP_ENTRE_CHAMADAS)

    revisoes = [revisao_openai, revisao_sabia]
    resolucoes = [resolucao_openai, resolucao_sabia]

    rubrica, bruto_rubrica = avaliar_rubrica(
        q,
        validacao_estrutura,
        validacao_sympy,
        revisoes,
        resolucoes,
    )

    score = calcular_score_final(
        q,
        rubrica,
        validacao_estrutura,
        validacao_sympy,
        revisoes,
        resolucoes,
    )

    auditoria = {
        "validacao_estrutura": validacao_estrutura,
        "validacao_sympy": validacao_sympy,
        "revisoes": [r.model_dump() for r in revisoes],
        "resolucoes_cegas": [r.model_dump() for r in resolucoes],
        "rubrica": rubrica.model_dump(),
        "score": score,
        "brutos": {
            "revisor_openai": bruto_rev_openai,
            "revisor_sabia": bruto_rev_sabia,
            "resolvedor_openai": bruto_res_openai,
            "resolvedor_sabia": bruto_res_sabia,
            "rubrica": bruto_rubrica,
        },
    }
    return auditoria


In [ ]:
def processar_item(numero: int, item: dict) -> dict:
    print("=" * 100)
    print(f"QUESTAO {numero} | {item['discipline']} | {item['tema']} | {item['taxonomy_level']}")
    print("=" * 100)

    registro = {
        "numero": numero,
        "plano": item,
        "iteracoes": [],
    }

    try:
        q_atual, bruto_autor = gerar_questao(item)
        registro["questao_inicial"] = q_atual.model_dump()
        registro["bruto_autor"] = bruto_autor

        melhor_q = q_atual
        melhor_score = -1
        melhor_auditoria = None
        auditoria_anterior = None

        for revisao_num in range(1, MAX_REVISOES + 1):
            print("-" * 100)
            print(f"Revisao {revisao_num}/{MAX_REVISOES}")

            auditoria = auditar_iteracao(q_atual)
            score = auditoria["score"]

            print("Score final:", score["score_final"], "| Status:", score["status"])
            print("Penalidade:", score["penalidade"])

            registro["iteracoes"].append(
                {
                    "revisao": revisao_num,
                    "questao_entrada": q_atual.model_dump(),
                    "auditoria": auditoria,
                }
            )

            if score["score_final"] > melhor_score:
                melhor_score = score["score_final"]
                melhor_q = q_atual
                melhor_auditoria = auditoria

            if score["status"] == "APROVADA":
                print(f"Aprovada com score {score['score_final']}.")
                break

            comite, bruto_comite = comite_revisao(
                q_atual,
                item,
                auditoria["validacao_estrutura"],
                auditoria["validacao_sympy"],
                [ReviewReport.model_validate(r) for r in auditoria["revisoes"]],
                [ResolucaoCega.model_validate(r) for r in auditoria["resolucoes_cegas"]],
                RubricaPedagogica.model_validate(auditoria["rubrica"]),
                score,
                auditoria_anterior=auditoria_anterior,
            )

            registro["iteracoes"][-1]["comite"] = comite.model_dump()
            registro["iteracoes"][-1]["bruto_comite"] = bruto_comite

            q_atual = comite.questao_revisada
            auditoria_anterior = auditoria
            time.sleep(SLEEP_ENTRE_CHAMADAS)

        registro["questao_final"] = melhor_q.model_dump()
        registro["auditoria_final"] = melhor_auditoria
        registro["score_final"] = melhor_score
        registro["n_revisoes"] = len(registro["iteracoes"])
        registro["status_final"] = (
            "APROVADA" if melhor_score >= LIMIAR_APROVACAO else "MELHOR_VERSAO_DISPONIVEL"
        )

    except Exception as e:
        print("ERRO:", e)
        traceback.print_exc()
        registro["erro"] = str(e)
        registro["traceback"] = traceback.format_exc()
        registro["questao_final"] = {}
        registro["auditoria_final"] = {}
        registro["score_final"] = ""
        registro["n_revisoes"] = 0
        registro["status_final"] = "ERRO"

    return registro


def rodar_pipeline(plano: List[dict]) -> List[dict]:
    resultados = []

    if REINICIAR_GERACAO:
        for arquivo in [ARQ_JSON, ARQ_CSV]:
            if Path(arquivo).exists():
                Path(arquivo).unlink()
        print("Geracao reiniciada: resultados anteriores removidos.")

    if ARQ_JSON.exists():
        with open(ARQ_JSON, "r", encoding="utf-8") as f:
            resultados = json.load(f)

    ids_feitos = {r.get("numero") for r in resultados if r.get("numero") is not None}

    for numero, item in enumerate(plano, start=1):
        if numero in ids_feitos:
            print(f"Pulando questao {numero}: ja existe no JSON.")
            continue

        registro = processar_item(numero, item)
        resultados.append(registro)
        salvar_resultados(resultados)
        print("Salvo:", ARQ_JSON, ARQ_CSV)
        time.sleep(SLEEP_ENTRE_CHAMADAS)

    return resultados


## 19. Rodar geração de novas questões

Arquivos gerados:

- `questoes_geradas_pipeline_robusto.json`: registro completo das iterações;
- `questoes_geradas_pipeline_robusto.csv`: tabela final com as melhores versões.


In [ ]:
# Execute esta celula para rodar o pipeline.
# Sugestao: teste primeiro com N_QUESTOES = 2 ou 3 na celula de configuracao.

resultados = rodar_pipeline(plano_questoes)

print("Finalizado.")
print("JSON:", ARQ_JSON)
print("CSV:", ARQ_CSV)

if ARQ_CSV.exists():
    display(pd.read_csv(ARQ_CSV).head())
else:
    print("CSV ainda nao foi gerado. Veja a lista de resultados/erros abaixo.")
    display(pd.DataFrame(resultados))


## 20. Baixar resultados

Esta célula baixa os principais arquivos gerados no Colab.


In [ ]:
arquivos_para_baixar = [
    ARQUIVO_AVALIACOES,
    ARQUIVO_REVISAO_JSON,
    ARQUIVO_REVISAO_CSV,
    ARQUIVO_REVISADAS,
    ARQUIVO_COMPARACAO_PDF,
    ARQ_JSON,
    ARQ_CSV,
]

for arquivo in arquivos_para_baixar:
    if Path(arquivo).exists():
        print("Arquivo disponivel:", arquivo)
        if files is not None:
            files.download(str(arquivo))
